[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/umangbagadi03/build-your-own-coding-agent/blob/main/notebooks/11_token_budgets.ipynb)

# 11. Token Budgets

**Part 3: Context** · about 20 minutes

Everything an agent knows during a task lives in one list of messages, and every model has a limit on how much of that list it can read at once: its **context window**. Long before you hit that limit, you run into two other problems. Every token is paid for again on each step, and a model working through a huge context tends to use the information in it less reliably.

Part 3 is about managing that space. This first notebook is about **measuring** it. You can't manage what you don't measure, and most people are surprised by what's actually filling their agent's context.

**By the end you will have:**

- a `context_report` that shows where every token goes
- seen how much the tool definitions alone cost, on every single call
- watched one big file take over an agent's context
- a per-result budget that keeps it under control

In [1]:
import pathlib, urllib.request

if not pathlib.Path("llm_setup.py").exists():  # Colab opens one notebook at a time, so fetch the helper
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/umangbagadi03/build-your-own-coding-agent/main/notebooks/llm_setup.py", "llm_setup.py"
    )

from llm_setup import connect

llm = connect()

OFFLINE MODE: a scripted local server, not a real model.
The HTTP calls are real. The answers are canned for these lessons.
Token counts are rough estimates and timings are simulated.
Set GROQ_API_KEY (free tier) or LLM_PROVIDER=ollama to use a real model.


In [2]:
import json
import time

import requests

url = f"{llm.base_url}/chat/completions"

headers = {"Content-Type": "application/json"}
if llm.api_key:
    headers["Authorization"] = f"Bearer {llm.api_key}"


def chat(messages, **options):
    payload = {"model": llm.model, "messages": messages, **options}
    response = requests.post(url, headers=headers, json=payload, timeout=llm.timeout)
    response.raise_for_status()
    return response.json()

## 1. Estimating tokens

Each model family has its own tokenizer, so the only exact count is the one the provider reports in `usage`. While building an agent you need an estimate *before* you send anything, though. The usual rule of thumb is **about 4 characters per token for English**.

How good is that rule? Let's ask the model. We send three texts of similar length and compare the estimate with the real count. (In offline mode the server itself uses the 4-characters rule, so the comparison only means something on a real model.)

In [3]:
def estimate_tokens(text):
    return max(1, len(text) // 4)


samples = {
    "English": "The quick brown fox jumps over the lazy dog while the team reviews the release notes. " * 3,
    "Python":  "def total(items):\n    return sum(i['price'] * i['qty'] for i in items if i['qty'] > 0)\n" * 3,
    "Hindi":   "आज की बैठक में हमने अगले महीने की योजना पर चर्चा की और सभी कामों की समीक्षा की। " * 3,
}

baseline = chat([{"role": "user", "content": "Hi"}], max_tokens=1)["usage"]["prompt_tokens"]

print(f"{'text':<9}{'chars':>7}{'estimate':>10}{'real':>7}{'chars per token':>17}")
for name, text in samples.items():
    real = chat([{"role": "user", "content": "Hi" + text}], max_tokens=1)["usage"]["prompt_tokens"] - baseline
    print(f"{name:<9}{len(text):>7}{estimate_tokens(text):>10}{real:>7}{len(text) / max(real, 1):>17.1f}")

text       chars  estimate   real  chars per token


English      258        64     64              4.0


Python       261        65     67              3.9


Hindi        240        60     60              4.0


On real tokenizers, English usually comes close to the rule, code often needs more tokens, and Hindi and many other non-Latin scripts can need several times more. If your agent works with Hindi documents, the 4-characters rule can underestimate badly. Calibrate it once for your model and your kind of text.

For the rest of this notebook we use the estimate, because it's free and it behaves the same everywhere.

## 2. Where the tokens go

Here's a report that breaks a request down into parts. It's the most useful debugging tool in this whole Part.

In [4]:
def context_report(messages, tools):
    """Estimate the tokens used by each part of a request."""
    names = {}  # tool_call_id -> tool name, so each result can be labelled
    parts = {"system prompt": 0, "tool definitions": estimate_tokens(json.dumps(tools)) if tools else 0,
             "user messages": 0, "assistant messages": 0}
    for message in messages:
        role, content = message["role"], message.get("content") or ""
        if role == "system":
            parts["system prompt"] += estimate_tokens(content)
        elif role == "user":
            parts["user messages"] += estimate_tokens(content)
        elif role == "assistant":
            calls = message.get("tool_calls") or []
            for call in calls:
                names[call["id"]] = call["function"]["name"]
            parts["assistant messages"] += estimate_tokens(content + json.dumps(calls))
        elif role == "tool":
            key = f"results: {names.get(message['tool_call_id'], 'unknown')}"
            parts[key] = parts.get(key, 0) + estimate_tokens(content)
    return parts


def show_report(parts, title=""):
    total = sum(parts.values())
    print(f"{title}  (about {total:,} tokens)")
    for name, tokens in sorted(parts.items(), key=lambda item: -item[1]):
        if tokens:
            bar = "#" * max(1, round(40 * tokens / total))
            print(f"  {name:<22}{tokens:>7,}  {bar}")

## 3. The tax you pay on every call

Let's start with an agent that hasn't done anything yet. It has the system prompt and the five tools from notebook 10's agent v2, and a one-line task.

In [5]:
def tool(tool_name, description, /, **properties):
    return {"type": "function", "function": {"name": tool_name, "description": description, "parameters": {
        "type": "object", "properties": {k: {"type": "string", "description": v} for k, v in properties.items()},
        "required": list(properties)}}}


AGENT_V2_TOOLS = [
    tool("run_command", "Run a shell command in the project folder, sandboxed. Some commands need approval. "
                        "Output starts with the exit code. Use `python` to run Python.", command="The command"),
    tool("find", "Search the project's Python files for text. Returns file:line: text.", pattern="Text to find"),
    tool("read_file", "Read a file with line numbers, optionally only lines start to end.", path="File path"),
    tool("edit_file", "Replace text in an existing file. old must match exactly once, without line numbers.",
         path="File path", old="Exact text to replace", new="Replacement text"),
    tool("create_file", "Create a new file. Fails if the file exists.", path="File path", content="File content"),
]

SYSTEM_PROMPT = """You are a careful coding agent. You fix bugs in a small Python project.

How to work:
1. Reproduce the problem first by running the command from the bug report. Read the error.
2. Find and read the code involved. Read only the lines you need.
3. Fix it with edit_file, changing as little as possible.
4. Run the original command again to confirm the fix.
5. Add a test for the bug in a new file with create_file, then run all tests with `python -m unittest`.

You're done only when the original command works and all tests pass. Finish with a short summary of
the cause and the fix. Don't claim anything you didn't see in a tool result."""

empty = [{"role": "system", "content": SYSTEM_PROMPT},
         {"role": "user", "content": "Bug report: `python wordcount.py empty.txt` crashes."}]
parts = context_report(empty, AGENT_V2_TOOLS)
show_report(parts, "Before the agent does anything")

fixed_cost = parts["system prompt"] + parts["tool definitions"]
print(f"\nOver a 20-step run, the system prompt and tool definitions alone add up to about {fixed_cost * 20:,} tokens.")

Before the agent does anything  (about 586 tokens)
  tool definitions          414  ############################
  system prompt             159  ###########
  user messages              13  #

Over a 20-step run, the system prompt and tool definitions alone add up to about 11,460 tokens.


The tool definitions cost more than twice as much as the system prompt, and both are sent on **every** call. Real agents with 20 or 40 tools spend thousands of tokens per call before doing any work. That's why Part 4 covers *prompt caching*, which makes this repeated prefix much cheaper, and why serious agents keep their tool list short and their descriptions tight.

## 4. One big file takes over

Now a research agent, like the one in notebook 04, with one addition: a long meeting transcript, the kind of file real projects are full of. The answer isn't even in it.

In [6]:
speakers = ["Ravi", "Meera", "Arjun", "Sana", "Kabir"]
topics = ["the vendor call", "the dashboard colours", "Friday's demo", "the office wifi", "the hiring plan",
          "the API docs", "the test flakiness", "the cafeteria menu"]
transcript = "\n".join(
    f"[10:{i % 60:02d}] {speakers[i % 5]}: Quick update on {topics[i % 8]}, nothing blocking, will follow up in chat."
    for i in range(300)
)

NOTES = {
    "standup.md": "Standup notes. The launch deadline moved to March 14 because of the audit.",
    "owners.md": "Launch owner: Priya. Backend owner: Sam.",
    "lunch.md": "The team voted for dosa on Friday.",
    "meeting_transcript.md": transcript,
}
print(f"meeting_transcript.md: {len(transcript):,} characters, about {estimate_tokens(transcript):,} tokens")

meeting_transcript.md: 27,142 characters, about 6,785 tokens


In [7]:
TOOLS = [
    tool("list_files", "List the names of all note files."),
    tool("read_file", "Read one note file by its exact name.", path="The file name"),
]


def run_research(question, result_limit=None, verbose=True):
    """The notebook 04 loop, plus an optional limit on how big each tool result can be."""
    messages = [
        {"role": "system", "content": "Answer using the tools. Read the files you need, then reply briefly."},
        {"role": "user", "content": question},
    ]
    billed = 0
    for step in range(1, 10):
        data = chat(messages, tools=TOOLS)
        billed += (data.get("usage") or {}).get("prompt_tokens", 0)
        message = data["choices"][0]["message"]
        messages.append(message)
        if verbose:
            show_report(context_report(messages[:-1], TOOLS), f"Step {step} sends")
        if not message.get("tool_calls"):
            return {"answer": message.get("content"), "messages": messages, "billed": billed, "steps": step}
        for call in message["tool_calls"]:
            arguments = json.loads(call["function"]["arguments"] or "{}")
            if call["function"]["name"] == "list_files":
                result = json.dumps(sorted(NOTES))
            else:
                result = NOTES.get(arguments.get("path"), f"Error: no file named {arguments.get('path')!r}")
            result = cap(result, result_limit)
            messages.append({"role": "tool", "tool_call_id": call["id"], "content": result})
    return {"answer": None, "messages": messages, "billed": billed, "steps": step}


def cap(text, max_tokens):
    """Keep a result under max_tokens, and say clearly that it was cut."""
    if max_tokens is None or estimate_tokens(text) <= max_tokens:
        return text
    return (text[: max_tokens * 4] + f"\n[... cut: this result was about {estimate_tokens(text):,} tokens "
            f"and the limit is {max_tokens:,}. Ask for a smaller part if you need more.]")


unlimited = run_research("What is the launch deadline, and who owns the launch?")
print("\nAnswer:", unlimited["answer"])

Step 1 sends  (about 135 tokens)
  tool definitions          105  ###############################
  system prompt              17  #####
  user messages              13  ####


Step 2 sends  (about 174 tokens)
  tool definitions          105  ########################
  assistant messages         23  #####
  system prompt              17  ####
  results: list_files        16  ####
  user messages              13  ###


Step 3 sends  (about 210 tokens)
  tool definitions          105  ####################
  assistant messages         51  ##########
  system prompt              17  ###
  results: list_files        16  ###
  user messages              13  ##
  results: read_file          8  ##


Step 4 sends  (about 7,112 tokens)
  results: read_file      6,821  ######################################
  assistant messages        140  #
  tool definitions          105  #
  system prompt              17  #
  results: list_files        16  #
  user messages              13  #

Answer: The launch deadline is March 14, and the owner is Priya.


Look at the last report. The transcript is most of the context, it's paid for on every step after it was read, and it contributed nothing to the answer. This happens all the time in real agents: one log file, one generated file or one huge test output, and the context is mostly noise.

## 5. A budget per tool result

The simplest defence is a cap on every tool result. For command output we kept the start and the end (notebook 07), because errors live at the end. For documents, the start is usually most useful, so here we keep the start and say exactly how much is missing.

In [8]:
limited = run_research("What is the launch deadline, and who owns the launch?", result_limit=300, verbose=False)
show_report(context_report(limited["messages"][:-1], TOOLS), "Final step with a 300-token limit per result")
print("\nAnswer:", limited["answer"])

print(f"\nInput tokens billed, whole run:  no limit {unlimited['billed']:,}   with limit {limited['billed']:,}")

Final step with a 300-token limit per result  (about 654 tokens)
  results: read_file        363  ######################
  assistant messages        140  #########
  tool definitions          105  ######
  system prompt              17  #
  results: list_files        16  #
  user messages              13  #

Answer: The launch deadline is March 14, and the owner is Priya.

Input tokens billed, whole run:  no limit 7,993   with limit 1,464


In [9]:
# Checkpoint: the limit must cut the bill without losing the answer.
assert limited["billed"] < unlimited["billed"]
if llm.offline:
    assert "March 14" in limited["answer"] and "Priya" in limited["answer"]
print("Checkpoint passed.")

Checkpoint passed.


The cut message matters as much as the cut itself. "Ask for a smaller part if you need more" tells the model there's more to see and how to get it. With a `read_file` that takes line ranges, like notebook 06's, the model can then page through a big file when it really needs to.

A per-result cap only keeps each *piece* small, though. Over a long task the history keeps growing step after step, and eventually it won't fit no matter how small the pieces are. That's the next notebook.

## Recap

- Measure your context. `context_report` will surprise you.
- The system prompt and tool definitions are paid for on **every** call, so keep them tight.
- One big tool result can dominate the context for the rest of the run.
- Cap every result, keep the useful end, and tell the model what was cut.
- The 4-characters-per-token rule is rough, and much worse for scripts like Devanagari.

## Exercises

1. Add a context budget for the whole request: before each call, if the estimate is over 80% of a limit you choose, print a warning. What should the agent do when it fires?
2. Shorten the tool descriptions in `AGENT_V2_TOOLS` as far as you can without losing meaning. How many tokens did you save per call, and over a 20-step run?
3. Give `read_file` an `offset` and `limit`, put the line about the deadline at the *end* of the transcript, and see whether your model pages through it.

**Next:** [12. Compaction](12_compaction.ipynb). What to do when the history itself gets too big.